# Aqua Epoch Phase 1 — reproducible hybrid ML

This notebook trains two potability classifiers and two pH regressors using the shared pipeline in `ml/train.py`. The classifier uses all 18 non-target features or the UI's `Turbidity`, `Hardness`, and `TDS`; the regressor predicts pH from all non-pH/non-Potability features or those same three UI features.

Feature missingness is retained and imputed inside each sklearn Pipeline. The validation ranges emitted by the run are observed training-data ranges only; they are not regulatory limits and do not certify water safety.

In [ ]:
# Colab/local: install dependencies once. The command is cwd-independent.
%pip install -q pandas numpy scikit-learn joblib

In [ ]:
from pathlib import Path
import sys

DATASET_FILENAME = 'Aqua_Epoch_BIS_Water_Quality_Dataset.xls - Aqua_Epoch_BIS_Water_Quality_Dataset.csv'
roots = [Path.cwd(), *Path.cwd().parents, Path('/content/Aqua_Epoch'), Path('/content')]
candidates = [root / DATASET_FILENAME for root in roots]
dataset_path = next((p for p in candidates if p.exists()), None)
if dataset_path is None:
    raise FileNotFoundError('Upload or mount the source CSV, then set dataset_path to its path.')
project_root = next((p for p in roots if (p / 'ml' / 'train.py').exists()), Path.cwd())
sys.path.insert(0, str(project_root))
from ml.train import train_models

artifact_dir = project_root / 'artifacts'
print('Dataset:', dataset_path)
print('Artifacts:', artifact_dir)

In [ ]:
manifest = train_models(dataset_path, artifact_dir, random_state=42, test_size=0.20, n_estimators=200)
manifest['cleaning_summary']

In [ ]:
# Compact evaluation view for the report/demo.
for name, details in manifest['models'].items():
    print(name, details['metrics'])

## Loading an artifact later

Each `.joblib` is a bundle with `pipeline` and `metadata`. The metadata includes feature order, cleaning summary, split seed, evaluation metrics, and advisory input-validation bounds. A backend should validate/request fields in that recorded order before calling the pipeline.

In [ ]:
from ml.train import load_artifact, validate_input_record

bundle = load_artifact(artifact_dir / 'classification_ui.joblib')
example = {'Turbidity': 3.2, 'Hardness': 150.0, 'TDS': 1200.0}
validation = validate_input_record(bundle, example)
validation